# 06 - Explainability and prediction pipeline (Phases 12 and 13) - run in VS CODE on your computer

**Needs:** the trained model folder in `models/xlm_roberta_hate/` (downloaded from Google Drive) and, installed locally:
```
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install -r requirements-model.txt
```
CPU is enough. Loading takes about a minute; one explanation takes a few seconds.

> Examples below can contain offensive wording because that is what the model is built to detect. Clear all outputs before committing.

## 1. Setup

In [ ]:
import sys, json, time
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import HTML, display
from src.utils.config import XLMR_DIR
assert (XLMR_DIR / "config.json").exists(), f"Model not found in {XLMR_DIR}. Download it from Google Drive first."
print("model folder:", XLMR_DIR, "->", sorted(p.name for p in XLMR_DIR.iterdir()))

## 2. Load the model once

In [ ]:
from src.model.predict import HateSpeechPredictor
from src.explainability.integrated_gradients import explain_text, render_html, attribution_bar
predictor = HateSpeechPredictor()
print("loaded on", predictor.device)

## 3. Phase 13 - the `predict(text)` function
Returns exactly: `prediction`, `confidence`, `language`, `important_words`.
`important_words` appear for HATE predictions only (they are the words with the highest attribution toward the predicted class).

In [ ]:
examples = [
    "I hate those disgusting people",
    "Have a wonderful day my friend",
    "\u0906\u092a \u0938\u092c \u0932\u094b\u0917 \u092c\u0939\u0941\u0924 \u0905\u091a\u094d\u091b\u0947 \u0939\u0948\u0902",
    "\u0b87\u0ba8\u0bcd\u0ba4 \u0bb5\u0bc0\u0b9f\u0bbf\u0baf\u0bcb \u0bae\u0bbf\u0b95\u0bb5\u0bc1\u0bae\u0bcd \u0b85\u0bb0\u0bc1\u0bae\u0bc8",
    "\u0d08 \u0d35\u0d40\u0d21\u0d3f\u0d2f\u0d4b \u0d35\u0d33\u0d30\u0d46 \u0d28\u0d28\u0d4d\u0d28\u0d3e\u0d2f\u0d3f\u0d1f\u0d4d\u0d1f\u0d41\u0d23\u0d4d\u0d1f\u0d4d",
]
for t in examples:
    print(t); print(json.dumps(predictor.predict(t), ensure_ascii=False, indent=2)); print()

## 4. Try your own text

In [ ]:
my_text = "type your own sentence here"
print(json.dumps(predictor.predict(my_text, details=True), ensure_ascii=False, indent=2))

## 5. Phase 12 - Integrated Gradients in detail
For one text: the prediction, a score for every word, a text bar, and the highlighted sentence.
`completeness_error` checks the explanation: the scores should add up to the change in the predicted probability
(small, roughly below 0.05, is good; if it is large, increase `n_steps`).

In [ ]:
def show(text, n_steps=50, explain_any_class=True):
    e = explain_text(text, predictor.model, predictor.tokenizer, n_steps=n_steps)
    print(f"Prediction: {e['predicted_label']}   confidence: {100 * e['confidence']:.1f}%   (explained class: {e['target_label']})")
    print(f"completeness error: {e['completeness_error']:.3f}  | n_steps: {e['n_steps']} | truncated: {e['truncated']}\n")
    for w in sorted(e["words"], key=lambda w: -w["importance"]):
        print(f"  {w['word']:<18} score {w['score']:+.3f}   importance {w['importance']:.2f}  {attribution_bar(w['importance'])}")
    print("\nHighest attribution toward the predicted class:", e["important_words"])
    display(HTML(render_html(e["words"])))
    return e

e = show("I hate those disgusting people")

## 6. How many steps are enough? (completeness error vs `n_steps`)

In [ ]:
rows = []
for n in (10, 25, 50, 100):
    t0 = time.time(); e = explain_text("I hate those disgusting people", predictor.model, predictor.tokenizer, n_steps=n)
    rows.append({"n_steps": n, "completeness_error": round(e["completeness_error"], 4), "seconds": round(time.time() - t0, 1),
                 "top_words": ", ".join(e["important_words"][:3])})
display(pd.DataFrame(rows))

## 7. Speed test on real test comments (optional)
Explains 15 test comments predicted as HATE. This shows how long a large-scale run would take (for that, use Colab).

In [ ]:
from src.data.io_utils import read_processed_csv
from src.utils.config import TEST_FILE
if TEST_FILE.exists():
    sample = read_processed_csv(TEST_FILE).sample(40, random_state=7)
    done, t0, out = 0, time.time(), []
    for _, r in sample.iterrows():
        res = predictor.predict(r["text"])
        if res["prediction"] == "HATE":
            out.append({"language": res["language"], "true_label": int(r["label"]), "confidence": res["confidence"], "important_words": ", ".join(res["important_words"])})
            done += 1
        if done == 15: break
    print(f"{done} explanations in {time.time() - t0:.0f} s")
    display(pd.DataFrame(out))
else:
    print("data/processed/test.csv not found - skip")

## 8. Wording for the report
Write: *"The explainability module identifies the tokens with the highest attribution scores toward the predicted hate class."*
Do **not** write: *"The model identified these words as hate words."*
Attributions describe how the model reached its output, not what is objectively hateful. See `docs/explainability.md`.